# 🌿 Air Quality Prediction - Complete Project
**Dataset:** Air Quality Data in India (2015-2020) - `city_day.csv`

**Is notebook mein sab kuch hai:**
1. Data load + EDA
2. Cleaning + feature engineering
3. Do tasks: **Current-day AQI estimate** aur **Next-day AQI forecast**
4. 3 models + baseline comparison (time-based split, no leakage)
5. Graphs, model save, Streamlit app, README

**Kaise chalana hai:** `city_day.csv` ko Colab ke Files panel mein upload karo, phir **Runtime > Run all**.

## 1. Libraries install aur import

In [ ]:
%pip install -q xgboost plotly streamlit pyngrok joblib

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

sns.set_style("whitegrid")

## 2. Data load

In [ ]:
files = glob.glob("**/city_day.csv", recursive=True)
if files:
    file_path = files[0]
else:
    from google.colab import files as colab_files   # sirf Colab mein chalega
    up = colab_files.upload()
    file_path = [n for n in up if n.lower().endswith(".csv")][0]

raw = pd.read_csv(file_path, parse_dates=["Date"])
print("File:", file_path, "| Shape:", raw.shape)
raw.head()

## 3. Data ko samjho (EDA)

In [ ]:
print("Date range:", raw.Date.min().date(), "to", raw.Date.max().date())
print("Cities:", raw.City.nunique())
print("Duplicate rows:", raw.duplicated().sum())
print("\nMissing values (%):")
print((raw.isna().mean() * 100).round(1).sort_values(ascending=False))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 5))

raw.groupby(raw.Date.dt.month)["AQI"].mean().plot(kind="bar", ax=ax[0], color="steelblue")
ax[0].set_title("Average AQI by Month"); ax[0].set_xlabel("Month")

raw.groupby("City")["AQI"].mean().sort_values(ascending=False).head(10).plot(
    kind="barh", ax=ax[1], color="indianred")
ax[1].set_title("Top 10 cities by average AQI"); ax[1].invert_yaxis()

sns.histplot(raw["AQI"].dropna(), bins=40, ax=ax[2], color="seagreen")
ax[2].set_title("AQI distribution")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
cols = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "AQI"]
sns.heatmap(raw[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap"); plt.show()

## 4. Cleaning aur features
- **Xylene** hata diya (61% khali)
- **AQI_Bucket** hata diya (ye AQI se hi bana hai, leakage hoga)
- Date se **Month** aur **DayOfWeek** nikala (`Year` nahi rakha, kyunki future mein kaam nahi karega)
- **City** ko one-hot encode kiya

In [ ]:
POLLUTANTS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene"]

df = raw.drop_duplicates().drop(columns=["Xylene", "AQI_Bucket"]).copy()
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek

CITIES = sorted(df["City"].unique())
city_dummies = pd.get_dummies(df["City"], prefix="City", dtype=int)
df = pd.concat([df, city_dummies], axis=1)
CITY_COLS = list(city_dummies.columns)

print("Rows:", len(df), "| Cities:", len(CITIES))

## 5. Do datasets banao

**Task A - Current-day estimate:** aaj ke pollutants se aaj ka AQI.
(Ye forecast nahi, estimate hai, kyunki AQI inhi pollutants se calculate hota hai.)

**Task B - Next-day forecast:** aaj ke pollutants + aaj ka AQI se **kal ka AQI**.
Har city ke andar agli date ka AQI shift karte hain, aur sirf consecutive din rakhte hain.

In [ ]:
# ---- Task A: current day ----
cur_features = POLLUTANTS + ["Month", "DayOfWeek"] + CITY_COLS
cur_data = df.dropna(subset=["AQI"]).copy()
print("Current-day rows:", len(cur_data))

# ---- Task B: next day ----
nxt = df.dropna(subset=["AQI"]).sort_values(["City", "Date"]).copy()
nxt["Next_Day_AQI"] = nxt.groupby("City")["AQI"].shift(-1)
nxt["Next_Date"] = nxt.groupby("City")["Date"].shift(-1)
nxt = nxt[nxt["Next_Date"] == nxt["Date"] + pd.Timedelta(days=1)].copy()   # sirf consecutive din

nxt_features = POLLUTANTS + ["AQI", "Month", "DayOfWeek"] + CITY_COLS
nxt_data = nxt
print("Next-day rows:", len(nxt_data))

## 6. Models train karo
- Time ke hisaab se **80:20 split** (purana data train, naya data test)
- Imputer aur scaler **pipeline ke andar** hain, to sirf train data se seekhte hain
- Models: Linear Regression, Random Forest, XGBoost

In [ ]:
def score(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
    }

def make_models():
    imp = lambda: SimpleImputer(strategy="median", keep_empty_features=True)
    return {
        "Linear Regression": Pipeline([("imputer", imp()), ("scaler", StandardScaler()),
                                       ("model", LinearRegression())]),
        "Random Forest": Pipeline([("imputer", imp()),
                                   ("model", RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                                                   random_state=42, n_jobs=-1))]),
        "XGBoost": Pipeline([("imputer", imp()),
                             ("model", XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                                                    subsample=0.8, colsample_bytree=0.8,
                                                    random_state=42, n_jobs=-1))]),
    }

def run_task(data, features, target, title):
    data = data.sort_values("Date").reset_index(drop=True)
    split = int(len(data) * 0.80)
    X, y = data[features], data[target]
    Xtr, Xte, ytr, yte = X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]
    dates = data["Date"]
    print(f"\n=== {title} ===")
    print(f"Train: {len(Xtr)} rows ({dates.iloc[0].date()} to {dates.iloc[split-1].date()})")
    print(f"Test : {len(Xte)} rows ({dates.iloc[split].date()} to {dates.iloc[-1].date()})")

    fitted, preds, results = {}, {}, {}
    for name, model in make_models().items():
        model.fit(Xtr, ytr)
        preds[name] = model.predict(Xte)
        fitted[name] = model
        results[name] = score(yte, preds[name])
    return dict(fitted=fitted, preds=preds, results=results, Xtr=Xtr, Xte=Xte, ytr=ytr, yte=yte,
                train_period=f"{dates.iloc[0].date()} to {dates.iloc[split-1].date()}",
                test_period=f"{dates.iloc[split].date()} to {dates.iloc[-1].date()}",
                test_dates=dates.iloc[split:].reset_index(drop=True))

cur = run_task(cur_data, cur_features, "AQI", "TASK A: CURRENT-DAY AQI ESTIMATE")
cur_table = pd.DataFrame(cur["results"]).T.sort_values("RMSE")
cur_table.round(3)

In [ ]:
nxt_out = run_task(nxt_data, nxt_features, "Next_Day_AQI", "TASK B: NEXT-DAY AQI FORECAST")

# Baseline: "kal ka AQI = aaj ka AQI" (sabse simple tareeka)
nxt_out["results"]["Baseline (aaj = kal)"] = score(nxt_out["yte"], nxt_out["Xte"]["AQI"])

nxt_table = pd.DataFrame(nxt_out["results"]).T.sort_values("RMSE")
nxt_table.round(3)

## 7. Graphs

In [ ]:
def plot_task(out, table, title):
    best = table.index[table.index != "Baseline (aaj = kal)"][0]
    fig, ax = plt.subplots(1, 3, figsize=(20, 5))

    table["R2"].plot(kind="bar", ax=ax[0], color="seagreen")
    ax[0].set_title(f"{title}: R2 by model"); ax[0].tick_params(axis="x", rotation=20)

    ax[1].scatter(out["yte"], out["preds"][best], alpha=0.35, s=10)
    lim = max(out["yte"].max(), out["preds"][best].max())
    ax[1].plot([0, lim], [0, lim], "k--")
    ax[1].set_xlabel("Actual AQI"); ax[1].set_ylabel("Predicted AQI")
    ax[1].set_title(f"Actual vs Predicted ({best})")

    xgb = out["fitted"]["XGBoost"]
    imp = pd.Series(xgb.named_steps["model"].feature_importances_, index=out["Xtr"].columns)
    imp.sort_values(ascending=False).head(10).sort_values().plot(kind="barh", ax=ax[2], color="darkorange")
    ax[2].set_title("Top 10 important features (XGBoost)")
    plt.tight_layout(); plt.show()

plot_task(cur, cur_table, "Current-day")
plot_task(nxt_out, nxt_table, "Next-day")

In [ ]:
# Delhi ka actual vs predicted (next-day, test period)
mask = (nxt_out["Xte"]["City_Delhi"] == 1).values
d_dates = nxt_out["test_dates"][mask]
plt.figure(figsize=(13, 4))
plt.plot(d_dates, nxt_out["yte"].values[mask], label="Actual")
plt.plot(d_dates, nxt_out["preds"]["XGBoost"][mask], label="Predicted (XGBoost)")
plt.title("Delhi: Next-day AQI, actual vs predicted (test period)")
plt.legend(); plt.xticks(rotation=30); plt.show()

## 8. Nateeja
- **Current-day** R2 zyada aata hai kyunki AQI usi din ke pollutants se banta hai (ye estimate hai).
- **Next-day** asli forecast hai. Baseline (aaj = kal) se model behtar hai, ye upar table mein dikh raha hai.
- Test period mein COVID lockdown (Mar 2020 se) bhi hai, jisse pollution achanak gira. Isliye test thoda mushkil hai.
- Deploy ke liye **XGBoost** chuna hai: accuracy Random Forest ke lagbhag barabar, file bahut chhoti (GitHub aur Streamlit Cloud ke liye zaroori).

## 9. Models aur metadata save karo

In [ ]:
import joblib
os.makedirs("saved_models", exist_ok=True)

joblib.dump(cur["fitted"]["XGBoost"], "saved_models/current_day_model.pkl", compress=3)
joblib.dump(nxt_out["fitted"]["XGBoost"], "saved_models/next_day_model.pkl", compress=3)

defaults = df[POLLUTANTS].median().round(1).to_dict()
defaults["AQI"] = float(df["AQI"].median().round(0))

meta = {
    "cities": CITIES,
    "pollutants": POLLUTANTS,
    "defaults": defaults,
    "current_features": cur_features,
    "next_features": nxt_features,
    "train_period": cur["train_period"],
    "test_period": cur["test_period"],
    "metrics": {
        "current_day": {k: {m: round(float(v), 4) for m, v in r.items()} for k, r in cur["results"].items()},
        "next_day": {k: {m: round(float(v), 4) for m, v in r.items()} for k, r in nxt_out["results"].items()},
    },
}
with open("saved_models/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

for fn in sorted(os.listdir("saved_models")):
    print(f"{fn:28s} {os.path.getsize('saved_models/' + fn) / 1024:8.0f} KB")

## 10. requirements.txt (exact versions, taaki deploy mein model load ho jaye)

In [ ]:
from importlib.metadata import version
pkgs = ["streamlit", "pandas", "numpy", "scikit-learn", "xgboost", "joblib", "plotly"]
with open("requirements.txt", "w") as f:
    for p in pkgs:
        f.write(f"{p}=={version(p)}\n")
print(open("requirements.txt").read())

## 11. Streamlit app (app.py)

In [ ]:
%%writefile app.py
# Air Quality Prediction - Streamlit App
import json
import os
from datetime import date

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Air Quality Prediction", page_icon="🌿",
                   layout="wide", initial_sidebar_state="expanded")

MODEL_DIR = "saved_models"

st.markdown("""
<style>
.hero-section {
    background: linear-gradient(120deg, #103C42, #087E75);
    padding: 36px; border-radius: 18px; color: white; margin-bottom: 24px;
}
.hero-section h1 { color: white !important; font-size: 36px; margin: 6px 0; }
.hero-section p { color: #D7F3EC; font-size: 16px; margin: 2px 0; }
div.stButton > button { border-radius: 10px; font-weight: 600; }
</style>
""", unsafe_allow_html=True)


# ---------------- LOAD MODELS ----------------
@st.cache_resource
def load_assets():
    current_model = joblib.load(os.path.join(MODEL_DIR, "current_day_model.pkl"))
    next_model = joblib.load(os.path.join(MODEL_DIR, "next_day_model.pkl"))
    with open(os.path.join(MODEL_DIR, "metadata.json")) as f:
        meta = json.load(f)
    return current_model, next_model, meta


try:
    current_model, next_model, meta = load_assets()
except Exception as e:
    st.error("Models load nahi ho paaye.")
    st.code(str(e))
    st.info("Pehle notebook run karo taaki `saved_models/` folder ban jaye "
            "(current_day_model.pkl, next_day_model.pkl, metadata.json).")
    st.stop()

CITIES = meta["cities"]
POLLUTANTS = meta["pollutants"]
DEFAULTS = meta["defaults"]
METRICS = meta["metrics"]


# ---------------- HELPERS ----------------
def get_category(aqi):
    if aqi <= 50:
        return "Good"
    if aqi <= 100:
        return "Satisfactory"
    if aqi <= 200:
        return "Moderate"
    if aqi <= 300:
        return "Poor"
    if aqi <= 400:
        return "Very Poor"
    return "Severe"


ADVICE = {
    "Good": "Air quality achhi hai. Bahar ki activities aaram se kar sakte ho.",
    "Satisfactory": "Sensitive logon ko halki saans ki takleef ho sakti hai.",
    "Moderate": "Sensitive log (bachche, bujurg, asthma) lambi outdoor activity kam karein.",
    "Poor": "Lambe samay tak bahar rehne se bachein, zaroorat ho to mask lagayein.",
    "Very Poor": "Bahar kam nikle aur health precautions follow karein.",
    "Severe": "Outdoor physical activity avoid karein aur local health advisory follow karein.",
}

COLORS = {"Good": "🟢", "Satisfactory": "🟡", "Moderate": "🟠",
          "Poor": "🔴", "Very Poor": "🟣", "Severe": "🟤"}


def build_row(features, values, city, the_date, today_aqi=None):
    """Model ke liye ek row banata hai. Missing pollutant = NaN (imputer sambhal lega)."""
    row = {f: 0.0 for f in features}
    for p in POLLUTANTS:
        if p in row:
            row[p] = values[p]
    row["Month"] = the_date.month
    row["DayOfWeek"] = the_date.weekday()
    city_col = "City_" + city
    if city_col in row:
        row[city_col] = 1.0
    if today_aqi is not None and "AQI" in row:
        row["AQI"] = today_aqi
    return pd.DataFrame([row], columns=features)


def metrics_table(task):
    return pd.DataFrame(METRICS[task]).T.reset_index().rename(columns={"index": "Model"})


# ---------------- SIDEBAR ----------------
st.sidebar.title("🌿 Air Quality Prediction")
st.sidebar.caption("Machine Learning Project")
page = st.sidebar.radio("Navigation", [
    "Dashboard", "Prediction", "Analytics", "Model Performance", "Health Recommendations"])
st.sidebar.divider()
st.sidebar.caption("Data: Air Quality Data in India (2015-2020)")


# ---------------- DASHBOARD ----------------
if page == "Dashboard":
    st.markdown("""
    <div class="hero-section">
        <p>MACHINE LEARNING PROJECT</p>
        <h1>🌿 Air Quality Prediction</h1>
        <p>Aaj ka AQI estimate karo. Kal ka AQI forecast karo.</p>
    </div>
    """, unsafe_allow_html=True)

    best_cur = METRICS["current_day"]["XGBoost"]
    best_next = METRICS["next_day"]["XGBoost"]
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Cities", len(CITIES))
    c2.metric("Current-day R²", f"{best_cur['R2']:.3f}")
    c3.metric("Next-day R²", f"{best_next['R2']:.3f}")
    c4.metric("Next-day MAE", f"{best_next['MAE']:.1f} AQI")

    st.divider()
    st.subheader("Project Overview")
    st.write("""
    Is project mein historical air pollution data (2015-2020, 26 Indian cities) se
    machine learning models train kiye gaye hain.

    - **Current-day estimate:** aaj ke pollutant readings se aaj ka AQI
    - **Next-day forecast:** aaj ke pollutants aur AQI se kal ka AQI
    - Split time ke hisaab se kiya gaya hai (purana data train, naya data test), isliye results honest hain
    """)
    st.warning("Ye historical data par bana ML demonstration hai, live AQI monitoring service nahi.")


# ---------------- PREDICTION ----------------
elif page == "Prediction":
    st.title("🌍 Air Quality Prediction")

    c1, c2 = st.columns(2)
    city = c1.selectbox("City", CITIES, index=CITIES.index("Delhi") if "Delhi" in CITIES else 0)
    the_date = c2.date_input("Date", date.today())

    mode = st.radio("Kya chahiye?", ["Current-day AQI estimate", "Next-day AQI forecast", "Dono"],
                    horizontal=True)

    st.subheader("Pollution Measurements (µg/m³; CO mg/m³)")
    missing = st.multiselect("Jin pollutants ki value pata nahi hai, unhe yahan chuno",
                             POLLUTANTS,
                             help="Inki value model khud estimate kar lega (training jaisa).")

    values = {}
    cols = st.columns(3)
    for i, p in enumerate(POLLUTANTS):
        with cols[i % 3]:
            if p in missing:
                st.number_input(p, value=float(DEFAULTS[p]), disabled=True, key="in_" + p)
                values[p] = np.nan
            else:
                values[p] = st.number_input(p, min_value=0.0, value=float(DEFAULTS[p]),
                                            step=1.0, key="in_" + p)

    known_today = False
    today_aqi = None
    if mode != "Current-day AQI estimate":
        known_today = st.checkbox("Mujhe aaj ka asli AQI pata hai")
        if known_today:
            today_aqi = st.number_input("Aaj ka AQI", min_value=0.0, max_value=500.0,
                                        value=float(DEFAULTS["AQI"]), step=1.0)

    if st.button("Predict Air Quality", type="primary"):
        try:
            cur_row = build_row(meta["current_features"], values, city, the_date)
            current_aqi = max(0.0, float(current_model.predict(cur_row)[0]))

            results = {}
            if mode in ("Current-day AQI estimate", "Dono"):
                results["Current Day"] = current_aqi

            if mode in ("Next-day AQI forecast", "Dono"):
                base_aqi = today_aqi if known_today else current_aqi
                nxt_row = build_row(meta["next_features"], values, city, the_date, base_aqi)
                results["Next Day"] = max(0.0, float(next_model.predict(nxt_row)[0]))

            st.success("Prediction ho gayi!")
            cols = st.columns(len(results))
            for col, (label, val) in zip(cols, results.items()):
                cat = get_category(val)
                col.metric(f"{label} AQI", f"{val:.0f}")
                col.subheader(f"{COLORS[cat]} {cat}")
                col.caption(ADVICE[cat])

            if "Next Day" in results and not known_today:
                st.info("Next-day forecast mein aaj ka AQI model ne khud estimate kiya hai. "
                        "Asli AQI pata ho to upar tick karke daalo, forecast behtar hoga.")

            mae_cur = METRICS["current_day"]["XGBoost"]["MAE"]
            mae_nxt = METRICS["next_day"]["XGBoost"]["MAE"]
            st.caption(f"Model ki typical galti: current-day ±{mae_cur:.0f} AQI, "
                       f"next-day ±{mae_nxt:.0f} AQI (test data par MAE).")

            if len(results) > 1:
                chart_df = pd.DataFrame({"Day": list(results.keys()), "AQI": list(results.values())})
                fig = px.bar(chart_df, x="Day", y="AQI", color="Day", title="Predicted AQI")
                st.plotly_chart(fig)
        except Exception as e:
            st.error(f"Prediction error: {e}")


# ---------------- ANALYTICS ----------------
elif page == "Analytics":
    st.title("📊 Air Quality Analytics")

    df = None
    if os.path.exists("city_day.csv"):
        df = pd.read_csv("city_day.csv")
    else:
        up = st.file_uploader("city_day.csv upload karo", type=["csv"])
        if up is not None:
            df = pd.read_csv(up)

    if df is None:
        st.info("Analytics dekhne ke liye city_day.csv upload karo.")
    else:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        st.subheader("Dataset Preview")
        st.dataframe(df.head(10))

        st.subheader("AQI Distribution")
        st.plotly_chart(px.histogram(df.dropna(subset=["AQI"]), x="AQI", nbins=40))

        st.subheader("Top 15 Cities by Average AQI")
        city_avg = (df.groupby("City")["AQI"].mean().reset_index()
                    .sort_values("AQI", ascending=False).head(15))
        st.plotly_chart(px.bar(city_avg, x="City", y="AQI"))

        st.subheader("Month-wise Average AQI")
        month_avg = df.dropna(subset=["AQI", "Date"]).groupby(df["Date"].dt.month)["AQI"].mean().reset_index()
        month_avg.columns = ["Month", "AQI"]
        st.plotly_chart(px.bar(month_avg, x="Month", y="AQI"))

        st.subheader("AQI Trend for a City")
        sel = st.selectbox("City chuno", sorted(df["City"].dropna().unique()))
        trend = df[(df["City"] == sel)].dropna(subset=["Date", "AQI"]).sort_values("Date")
        st.plotly_chart(px.line(trend, x="Date", y="AQI", title=f"{sel}: AQI over time"))


# ---------------- MODEL PERFORMANCE ----------------
elif page == "Model Performance":
    st.title("📈 Model Performance")
    st.write(f"Train: {meta['train_period']}  |  Test: {meta['test_period']} "
             "(time ke hisaab se split, random nahi)")

    for task, title in [("current_day", "Current-Day AQI Estimate"),
                        ("next_day", "Next-Day AQI Forecast")]:
        st.subheader(title)
        tbl = metrics_table(task)
        st.dataframe(tbl.round(3), hide_index=True)
        st.plotly_chart(px.bar(tbl, x="Model", y="R2", color="Model", title=f"{title}: R² score"))

    st.info("Baseline (aaj = kal) wo simple tareeka hai jisme aaj ka AQI hi kal ka AQI maan liya jaata hai. "
            "Model ko isse behtar hona chahiye.")
    st.caption("Test period (Jan-Jul 2020) mein COVID lockdown tha, jisse pollution achanak gira. "
               "Isliye test thoda mushkil hai.")
    st.write("Deployed models:", type(current_model.named_steps['model']).__name__,
             "(current) /", type(next_model.named_steps['model']).__name__, "(next-day)")


# ---------------- HEALTH RECOMMENDATIONS ----------------
elif page == "Health Recommendations":
    st.title("💚 Health Recommendations")
    ranges = [("Good", "0-50"), ("Satisfactory", "51-100"), ("Moderate", "101-200"),
              ("Poor", "201-300"), ("Very Poor", "301-400"), ("Severe", "401+")]
    for name, rng in ranges:
        with st.expander(f"{COLORS[name]} {name} AQI ({rng})"):
            st.write(ADVICE[name])
    st.warning("Ye general precautions hain, personal medical advice nahi.")


## 12. README.md (apne aap results ke saath banta hai)

In [ ]:
c = meta["metrics"]["current_day"]["XGBoost"]
n = meta["metrics"]["next_day"]["XGBoost"]
b = meta["metrics"]["next_day"]["Baseline (aaj = kal)"]

readme = f"""# Air Quality Prediction (India)

Machine learning project jo Indian cities ke AQI ko do tarah se predict karta hai:
1. **Current-day estimate:** aaj ke pollutant readings se aaj ka AQI
2. **Next-day forecast:** aaj ke pollutants aur AQI se kal ka AQI

## Dataset
[Air Quality Data in India (2015-2020)](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india) - `city_day.csv`, 26 cities.

## Approach
- Cleaning: duplicates check, Xylene (61% missing) drop, AQI_Bucket drop (leakage)
- Features: 11 pollutants, Month, DayOfWeek, City (one-hot); next-day task mein aaj ka AQI bhi
- Next-day target: har city ke andar agli date ka AQI (sirf consecutive din)
- **Time-based split** (80:20): train {meta["train_period"]}, test {meta["test_period"]}
- Imputer aur scaler pipeline ke andar (train data se hi fit)
- Models: Linear Regression, Random Forest, XGBoost (XGBoost deployed)

## Results (test set)
| Task | Model | MAE | RMSE | R2 |
|---|---|---|---|---|
| Current-day | XGBoost | {c["MAE"]:.2f} | {c["RMSE"]:.2f} | {c["R2"]:.3f} |
| Next-day | XGBoost | {n["MAE"]:.2f} | {n["RMSE"]:.2f} | {n["R2"]:.3f} |
| Next-day | Baseline (aaj = kal) | {b["MAE"]:.2f} | {b["RMSE"]:.2f} | {b["R2"]:.3f} |

Note: test period mein COVID-19 lockdown (2020) shamil hai, jisse pollution achanak gira.

## Run locally
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Files
- `air_quality_project.ipynb` - poora notebook (EDA, training, evaluation)
- `app.py` - Streamlit web app
- `saved_models/` - trained models aur metadata
- `requirements.txt`

## Limitations
- Historical data par bana demo hai, live AQI service nahi
- Sirf 2015-2020 ka data; naye saalon mein pattern badal sakte hain
"""
open("README.md", "w", encoding="utf-8").write(readme)
print(readme[:1200])

## 13. App chalao (Colab mein)
1. **ngrok.com** par free account banao, **Your Authtoken** copy karo
2. Neeche `PASTE_YOUR_TOKEN` ki jagah wo token daalo
3. Cell chalao, jo link aaye usse kholo (agar "Visit Site" button aaye to dabao)

*Ye link Colab band hone par band ho jata hai. Permanent link ke liye GitHub + Streamlit Community Cloud use karo.*

In [ ]:
from pyngrok import ngrok
import subprocess, time

ngrok.kill()
ngrok.set_auth_token("PASTE_YOUR_TOKEN")

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
print("Your app:", ngrok.connect(8501).public_url)